# Knowledge Base QA — Colab Training Notebook ★ H100 AUTOPILOT, RESUME-SAFE, FLEXIBLE-GPU

Trains the **Knowledge Base QA (RAG)** system (Project #3) on Google Colab.

✅ **One button** = build KB index → fine-tune retriever + reader (resume-safe) → evaluate → benchmark → error analysis → faithfulness → **auto-generate `report.pdf` + `slides.pptx`**
✅ **Survives disconnects**: artifacts + checkpoints live on Google Drive; re-run → training **resumes**.
✅ **Flexible GPU**: auto-detects **H100 / A100 / L4 / T4** and picks safe batch sizes + precision.
✅ **Colab-safe deps**: never reinstalls torch.

> Set `GIT_REPO_URL` in **Controls**, then `Runtime → Run all`.


In [ ]:
#@title 0) Controls  { display-mode: "form" }
GIT_REPO_URL = "https://github.com/ledinhminhquan/03_Knowledge_Base_QA"  #@param {type:"string"}#  ^ Your GitHub repo URL (e.g. https://github.com/<you>/kbqa.git). Or use DRIVE_REPO_DIR below.
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_PROJECT_DIR = "NLP_Project/kbqa"  #@param {type:"string"}
DRIVE_REPO_DIR = ""  #@param {type:"string"}

# MODELS
RETRIEVER_MODEL = "BAAI/bge-base-en-v1.5"  #@param ["BAAI/bge-base-en-v1.5", "intfloat/e5-base-v2", "sentence-transformers/all-MiniLM-L6-v2"]
READER_MODEL = "deepset/roberta-base-squad2"  #@param ["deepset/roberta-base-squad2", "deepset/deberta-v3-large-squad2", "distilbert-base-cased-distilled-squad"]

# PIPELINE TOGGLES
RUN_AUTOPILOT = True   #@param {type:"boolean"}
TRAIN_RETRIEVER = True #@param {type:"boolean"}
TRAIN_READER = True    #@param {type:"boolean"}
DEBUG_LIMIT = 0        #@param {type:"integer"}
#  ^ 0 = full data. Set e.g. 2000 for a fast smoke test.

PROJECT_TITLE = "Knowledge Base Question-Answering System"  #@param {type:"string"}
AUTHOR = "Le Dinh Minh Quan"  #@param {type:"string"}
print("Controls set. retriever:", RETRIEVER_MODEL, "| reader:", READER_MODEL)


In [ ]:
#@title 1) Check GPU
!nvidia-smi -L
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


In [ ]:
#@title 2) Mount Drive + artifact paths & HF caches (before importing torch)
import os
from pathlib import Path
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR
else:
    BASE = Path("/content/kbqa_artifacts")

ART = BASE / "artifacts"
for sub in ("data", "models", "index", "runs"):
    (ART / sub).mkdir(parents=True, exist_ok=True)
(BASE / "hf_cache").mkdir(parents=True, exist_ok=True)

os.environ["KBQA_ARTIFACTS_DIR"] = str(ART)
os.environ["KBQA_MODEL_DIR"]     = str(ART / "models")
os.environ["KBQA_INDEX_DIR"]     = str(ART / "index")
os.environ["KBQA_RUN_DIR"]       = str(ART / "runs")
os.environ["KBQA_DATA_DIR"]      = str(ART / "data")
os.environ["HF_HOME"]            = str(BASE / "hf_cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Artifacts ->", ART)


In [ ]:
#@title 3) Preflight (Drive writable + disk)
import os, shutil
from pathlib import Path
art = Path(os.environ["KBQA_ARTIFACTS_DIR"])
try:
    t = art / ".w"; t.write_text("ok"); t.unlink(); print("[OK] writable:", art)
except Exception as e:
    print("[FAIL]", e)
_,_,free = shutil.disk_usage("/content"); print(f"[INFO] /content free: {free/1e9:.1f} GB")


In [ ]:
#@title 4) Get the project source (git clone, or copy from Drive)
import os, shutil, subprocess
from pathlib import Path
REPO = Path("/content/kbqa_repo")
def _ok(p): return (p/"pyproject.toml").exists() and (p/"src"/"kbqa").exists()
if _ok(REPO): print("[OK] repo present")
elif GIT_REPO_URL.strip():
    if REPO.exists(): shutil.rmtree(REPO)
    subprocess.run(["git","clone","--depth","1",GIT_REPO_URL.strip(),str(REPO)], check=True)
elif DRIVE_REPO_DIR.strip():
    shutil.copytree(Path("/content/drive/MyDrive")/DRIVE_REPO_DIR.strip(), REPO, dirs_exist_ok=True)
else:
    raise SystemExit("Set GIT_REPO_URL (recommended) or DRIVE_REPO_DIR in Controls.")
assert _ok(REPO), f"repo incomplete at {REPO}"
os.chdir(REPO); print("[OK] cwd:", os.getcwd())


In [ ]:
#@title 5) Install dependencies (Colab-safe: never touch torch)
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements_colab.txt"], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",".","--no-deps"], check=True)
print("[OK] deps installed (torch untouched)")


In [ ]:
#@title 6) Verify env + enable performance knobs
import torch, transformers, datasets, sentence_transformers
print("torch:", torch.__version__, "| cuda:", torch.version.cuda)
print("transformers:", transformers.__version__, "| sentence-transformers:", sentence_transformers.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print("[WARN] No GPU — use Runtime -> Change runtime type -> GPU.")


In [ ]:
#@title 7) Auto GPU profile (batch sizes + precision)
import torch
prof = {"reader_batch": 8, "retriever_batch": 32, "bf16": False, "fp16": True}
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0).upper(); bf16 = torch.cuda.is_bf16_supported()
    if "H100" in name or "A100" in name: rb, qb = 48, 128
    elif "L4" in name or "L40" in name or "A10" in name: rb, qb = 24, 96
    elif "T4" in name or "V100" in name: rb, qb = 12, 48
    else: rb, qb = 16, 64
    if "large" in READER_MODEL.lower(): rb = max(4, rb // 3)
    prof = {"reader_batch": rb, "retriever_batch": qb, "bf16": bool(bf16), "fp16": (not bf16)}
print("GPU profile:", prof)


In [ ]:
#@title 8) Write the Colab training config (configs/train_colab.yaml)
import yaml
from pathlib import Path
cfg = yaml.safe_load(Path("configs/train.yaml").read_text(encoding="utf-8"))
cfg["project_title"] = PROJECT_TITLE; cfg["author"] = AUTHOR
cfg["retriever"]["bi_encoder_model"] = RETRIEVER_MODEL
cfg["retriever"]["e5_style"] = ("e5" in RETRIEVER_MODEL.lower())
cfg["retriever"]["train_batch_size"] = prof["retriever_batch"]
cfg["reader"]["model_name"] = READER_MODEL
cfg["reader"]["per_device_train_batch_size"] = prof["reader_batch"]
cfg["reader"]["bf16"] = prof["bf16"]; cfg["reader"]["fp16"] = prof["fp16"]
# Use the bigger GPU reranker when a strong GPU is present.
import torch
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    cfg["reranker"]["use_gpu_reranker"] = True
Path("configs/train_colab.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
print("Wrote configs/train_colab.yaml")
print(yaml.safe_dump({"retriever": cfg["retriever"]["bi_encoder_model"], "reader": cfg["reader"]["model_name"], **prof}))


In [ ]:
#@title 9) Build the demo knowledge base + sanity check
import os
os.environ["KBQA_INFER_CONFIG"] = "configs/train_colab.yaml"
limit = DEBUG_LIMIT if DEBUG_LIMIT > 0 else None
!python -m kbqa.cli build-kb --config configs/train_colab.yaml {("--limit " + str(limit)) if limit else ""}


In [ ]:
#@title 10) ★ ONE BUTTON: autopilot (resume-safe). Re-run after a disconnect to RESUME.
import subprocess, sys
if RUN_AUTOPILOT:
    cmd = [sys.executable,"-m","kbqa.cli","autopilot","--config","configs/train_colab.yaml",
           "--title",PROJECT_TITLE,"--author",AUTHOR]
    if DEBUG_LIMIT > 0: cmd += ["--limit", str(DEBUG_LIMIT)]
    if not (TRAIN_RETRIEVER or TRAIN_READER): cmd += ["--no-train"]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=False)
else:
    print("RUN_AUTOPILOT off — use the individual cells below.")


## Individual steps (optional) — all idempotent + resume-safe

In [ ]:
#@title 11a) Fine-tune the dense retriever (MNRL + hard negatives)
if TRAIN_RETRIEVER:
    !python -m kbqa.cli train-retriever --config configs/train_colab.yaml {("--limit " + str(DEBUG_LIMIT)) if DEBUG_LIMIT>0 else ""}
else: print("TRAIN_RETRIEVER off")


In [ ]:
#@title 11b) Fine-tune the extractive reader (SQuAD v2, resume-safe)
if TRAIN_READER:
    !python -m kbqa.cli train-reader --config configs/train_colab.yaml {("--limit " + str(DEBUG_LIMIT)) if DEBUG_LIMIT>0 else ""}
else: print("TRAIN_READER off")


In [ ]:
#@title 11c) Rebuild the KB index with the fine-tuned retriever, then evaluate
!python -m kbqa.cli build-kb  --config configs/train_colab.yaml {("--limit " + str(DEBUG_LIMIT)) if DEBUG_LIMIT>0 else ""}
!python -m kbqa.cli evaluate  --config configs/train_colab.yaml {("--limit " + str(DEBUG_LIMIT)) if DEBUG_LIMIT>0 else ""}


In [ ]:
#@title 12) Diagnostics: eval metrics + model metadata
import json, os
from pathlib import Path
run_dir = Path(os.environ["KBQA_RUN_DIR"])
evals = sorted(run_dir.glob("eval-*/eval.json"))
if evals:
    print("== Latest evaluation =="); print(json.dumps(json.loads(evals[-1].read_text()).get("summary", {}), indent=2))
for comp in ("retriever", "reader"):
    meta = Path(os.environ["KBQA_MODEL_DIR"]) / comp / "latest" / "model_metadata.json"
    if meta.exists():
        m = json.loads(meta.read_text()); print(f"\n== {comp} ==", m.get("base_model"), "| metrics:", m.get("metrics", {}))


## ✅ Test the trained model — ask grounded questions

In [ ]:
#@title 13) Ask the trained RAG agent (with citations) on the demo KB
from kbqa.config import load_config
from kbqa.agent.rag_agent import RAGAgent
cfg = load_config("configs/train_colab.yaml")
agent = RAGAgent(cfg)   # loads the fine-tuned retriever + reader + KB index from Drive
for q in ["What does FAISS stand for?",
          "Who created the Python programming language?",
          "What is the capital of France?"]:   # last one is NOT in the demo KB -> should abstain
    s = agent.ask(q)
    print(f"\nQ: {q}\n  status: {s.status}\n  answer: {s.answer}\n  conf={s.confidence:.2f} faithful={s.faithfulness:.2f}")
    if s.citations: print("  cite:", [c.get("title") or c.get("chunk_id") for c in s.citations])


In [ ]:
#@title 14) Locate the deliverables (report.pdf + slides.pptx + bundle)
import os
from pathlib import Path
sub = Path(os.environ["KBQA_ARTIFACTS_DIR"]) / "submission"
latest = sorted(sub.glob("submission-*"))
if latest:
    print("Submission folder:", latest[-1])
    for f in sorted(latest[-1].iterdir()):
        print("  -", f.name, f"({f.stat().st_size//1024} KB)" if f.is_file() else "")
else:
    print("Run the autopilot cell (10) first.")


In [ ]:
#@title 15) (Optional) Serve the API + demo (port 7860)
RUN_SERVER = False  #@param {type:"boolean"}
if RUN_SERVER:
    import os; os.environ["KBQA_INFER_CONFIG"] = "configs/train_colab.yaml"
    !python -m uvicorn kbqa.api.app_combined:app --host 0.0.0.0 --port 7860
else:
    print("Set RUN_SERVER=True to launch the API + demo UI.")


## Final checklists

**Deployment**
- [ ] `artifacts/models/retriever/latest/` and `artifacts/models/reader/latest/` exist (+ `model_metadata.json`)
- [ ] `artifacts/index/` has the FAISS index (`kbqa build-kb`)
- [ ] `kbqa evaluate` shows retrieval recall + answer F1; the agent **abstains** on out-of-KB questions
- [ ] API `/health` returns `index.n_vectors > 0`

**Submission**
- [ ] `report.pdf` · `slides.pptx` · `submission_bundle.zip` · `grading_checklist.json` (all PASS)
- [ ] Push the repo (models/ + artifacts/ git-ignored) to your public GitHub repo
